Reference: https://maartengr.github.io/BERTopic/index.html#installation

In [3068]:
from bertopic import BERTopic
import ssl
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from umap import UMAP
from matplotlib.lines import Line2D
import plotly
import plotly.io as pio
import plotly.graph_objects as go
import re
import plotly.colors as colors
from plotly.subplots import make_subplots
#remove stopwords from the topics list
import nltk
from nltk.corpus import stopwords

# Disable SSL verification - because it is annoying
ssl._create_default_https_context = ssl._create_unverified_context

In [3026]:
# Download stopwords (if not already downloaded)
nltk.download('stopwords')

# Get the English stopwords
stopwords = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/priyadcosta/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3027]:
'''
Load the datasets at join them convert the message column into strings. 
'''

df_csop = pd.read_csv('../output/csopII_output_chat_level.csv')
df_jury = pd.read_csv('../output/jury_output_chat_level.csv')

df_csop['message'] = df_csop['message'].astype(str)
df_jury['message'] = df_jury['message'].astype(str)


In [3028]:
'''
Fit the model based on the message column. Each message is a document 
@param
The list of documents/messages we want to fit the model on
'''
def fit_model(documents):
    #umap model initialized to random_state 2 to eliminate stochasticity 
    umap_model = UMAP(n_neighbors=15, n_components=5,min_dist=0.0, metric='cosine', random_state=2)
    model = BERTopic(top_n_words=30,umap_model=umap_model)
    model.fit_transform(documents)
    return model

In [3029]:
'''
Get the top n topics from the model
@param : The BERTopic model, the number of topics we want (This is excluding topic -1)
'''
def get_top_topics(topic_model,n):
    return_list = []
    for i in range(0, n): 
        topic_list = topic_model.get_topic(i)
        return_list.append(topic_list)  
    return return_list  

In [3030]:
'''
Divides the conversation into n equal chunks of time and adds a label for each chunk
@param: The dataframe, the desired number of chunks
'''
def create_chunks(df,num_chunks):
    # Convert timestamp column to DateTime format
    df['timestamp'] = pd.to_datetime(df['timestamp'])

    # Calculate the total duration of the conversation
    total_duration = (df['timestamp'].max() - df['timestamp'].min()).total_seconds()

    # Calculate the duration of each chunk
    chunk_duration = total_duration / num_chunks

    # Add a new column for chunk number
    df['chunk'] = -1 

    # Assign the chunk number for each row
    for index, row in df.iterrows():

        #get the timestamp 
        timestamp = row['timestamp']

        #calculate the chunk number
        chunk_number = int(((timestamp - df['timestamp'].min())).total_seconds() / chunk_duration)

        #restrict the range of the chunks from 0 to num_chunks - 1
        if chunk_number >= num_chunks:
            df.at[index, 'chunk'] = num_chunks - 1
        else:
            df.at[index, 'chunk'] = chunk_number

In [3031]:
'''
Extract the SBERT Embeddings from the model
'''
def extract_embeddings(model,df):
    embeddings_matrix = model._extract_embeddings(df['message'].tolist())
    return embeddings_matrix

'''
Get the representative docs for a topic and convert it to a df
'''
def get_rep_docs(model,topic_num):
    data = model.get_representative_docs(topic_num)
    df = pd.DataFrame(data,columns=['rep_docs'])
    return df

'''
Vectorize the documents for the topic
'''
def create_bert_vectors(df,on_column):
    docs = df[on_column].tolist()
    sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
    return sentence_model.encode(docs, show_progress_bar=False)

In [3032]:
'''
Get the embeddings for the top topics across the conversation
'''

def get_embeddings_top_topics(df,num_topics):

    #fit the model
    model = fit_model(df['message'].tolist())

    #get the top topics
    top_topics = get_top_topics(model,num_topics)
    
    embeddings_for_top_topics = []
    
    #for each topic
    for j in range(0,len(top_topics)):
        
        #get the representative documents 
        rep_docs_df = get_rep_docs(model,j)

        #create embeddings of the representative documents for each topic
        topic_embeddings = create_bert_vectors(rep_docs_df,'rep_docs')

        embeddings_for_top_topics.append(topic_embeddings) 
        
    return embeddings_for_top_topics

In [3033]:
'''
Get the embeddings for each chunk
'''
def get_embeddings_per_chunk(df,num_chunks):

    chunk_embeddings_list = []

    for i in range(0,num_chunks):
        convo = df[df['chunk'] == i]

        #if the df is empty, i.e there is a pause, create a pseudo 384-dimension embeddings with all values as 0
        if convo.empty:            
            #create n arrays of length 384 (number of dimensions)
            chunk_embeddings = np.zeros((1, 384))

        else:
            chunk_embeddings = create_bert_vectors(convo,'message')
        chunk_embeddings_list.append(chunk_embeddings)
    
    return chunk_embeddings_list

In [3034]:
#rows - every chunk, cols - topics 
def get_similarity(embeddings_per_chunk,embeddings_top_topics):

    topics_per_chunk = []
    for i in range (0,len(embeddings_per_chunk)):
        
        #calculate the cosine similarity for the topic and the chunk
        embeddings = np.mean(embeddings_per_chunk[i], axis=0) #get the average vectors for the chunk(if there are 4 docs, we get the average of 4 docs for each of the 384 dimensions)
        embeddings = np.nan_to_num(embeddings, nan=0) #replace any nan vectors with 0
        embeddings = embeddings.reshape(1, -1) #reshape to a 2D matrix (needed for the cosine similarity function)

        topics = []
        
        # 
        for j in range(0,len(embeddings_top_topics)):
            
            topic = np.mean(embeddings_top_topics[j], axis=0) #get the average vectors for the chunk(if there are 4 docs, we get the average of 4 docs for each of the 384 dimensions)
            topic = np.nan_to_num(topic, nan=0) #replace any nan vectors with 0
            topic = topic.reshape(1, -1) #reshape to a 2D matrix (needed for the cosine similarity function)

            #calculate the cosine similarity betwen the topic embeddings and the chunk embeddings
            cos = cosine_similarity(embeddings,topic)
            topics.append(cos[0][0])

        topics_per_chunk.append(topics)
    
    return topics_per_chunk
    

In [3035]:
'''
Remove the stop words from the topic to visualize the topics better
'''
def create_topics_without_stopwords(top_topics):
    cols = []
    for i in range(0,len(top_topics)):
        filtered_words = [word[0] for word in top_topics[i] if word[0].lower() not in stopwords]
        cols.append(filtered_words)

    topic_list = []
    for i in range(0,len(cols)):
        topic = ""
        for j in range(0,len(cols[i])):
            topic = topic + cols[i][j] +", "

        topic_list.append(topic)
    
    return topic_list

In [3093]:
'''
#THIS CODE WAS ORIGINALLY WRITTEN BY CHAT GPT, BECAUSE I WAS UNSURE ABOUT THE SYNTAX OF CREATING 
INTERACTIVE VISUALS. I THEN MODIFIED AND SIMPLIFIED IT TO SUIT OUR NEEDS
'''
def plot_with_plotly(df):  

    #Get the headings of the df eg. Convo 0 Topic 0, Convo 0 Topic 1. 
    # I am not changing the original DF so that it is still readable to us when we write it to csv
    convo_columns = df.columns 

    # Extract conversation columns and topics. For e.g, if we have a heading as Convo 0 Topic 1
    #col.split(' ')[-1] gives us the topic number as a string. This extracts the strings of unique topic numbers
    topics = sorted(list(set([col.split(' ')[-1] for col in convo_columns])))

    # Set colors for each topic dynamically - Not yet sure how it exactly works :(
    num_topics = len(topics)
    color_palette = plotly.colors.qualitative.D3
    topic_colors = {topic: color_palette[i % len(color_palette)] for i, topic in enumerate(topics)}

    #I HAVE VERFIED THIS LOGIC BY MANUALLY COMPUTING THE AVERAGES IN AN EXCEL SHEET
    '''
    ABOUT TRACES:

    In Plotly, a "trace" refers to a data object that describes how to visualize a set of data points. 
    It contains information about the type of plot (e.g., scatter, line, bar), the data to be plotted
    (x, y, z coordinates or categorical values), and various styling options such as line color, marker
    shape, and more.
    
    Each trace represents a distinct set of data or a specific aspect of the visualization. For example, 
    if you want to plot multiple lines on the same graph, you would create a separate trace for each line.

    Traces are typically added to a data list and then passed to the `go.Figure` object in Plotly. 
    The `go.Scatter` class is commonly used to create traces for line plots and scatter plots, but there
    are other trace classes available for different types of plots, such as `go.Bar` for bar plots and
    `go.Surface` for 3D surface plots.
    '''

    # Create traces for each conversation and topic
    traces = []
    for col in convo_columns:

        # Splits the original column headsers. For e.g., if the heading is Convo 0 Topic 1,
        convo = col.split(' ')[1] # This will be 0
        topic = col.split(' ')[-1] # This will be 1

        # Set visibility to True for convo 0 and False for others - This means the default value of the drop down is Convo 0
        # visible = True if convo == '0' else False  
        visible = True

        #create a trace object
        trace = go.Scatter(
            x=df.index,
            y=df[col],
            name=col,
            visible=visible,
            line=dict(color=topic_colors[topic], width=1)  # Set color based on topic
        )
        traces.append(trace)

    # Create trace for average of each topic
    avg_traces = []
    for topic in topics:
        '''
        get all the column headers which contain the current topic. For e.g. if the topic is '1', 
        and there are 2 convos. This list contain the headers: Convo 0 Topic 1,Convo 1 Topic 1.
        '''
        topic_columns = [col for col in convo_columns if col.endswith(topic)]

        '''
        Average of Topics Across Convos for each chunk. Axis is 1 as averaging across columns.
        Consider the following example df:

        This df has 3 rows indicating there are 3 chunks of time.
        data = {
            'Convo 0 Topic 0': [0.476381, 0.795204, 0.559577],
            'Convo 0 Topic 1': [0.466211, 0.498702, 0.768713],
            'Convo 1 Topic 0': [0.699774, 0.734668, 0.767569],
            'Convo 1 Topic 1': [0.449561, 0.484407, 0.523591]
        }

        If the current topic is topic 1, the code will average in the following way:

        Average for Chunk 0: (0.466211 + 0.449561) / 2
        Average for Chunk 1: (0.498702 + 0.484407) / 2
        Average for Chunk 2: (0.768713+ 0.523591) / 2

        Verfied by manual calculations in Excel
        '''
        avg_values = np.mean(df[topic_columns], axis=1)

        #create the trace object
        avg_trace = go.Scatter(
            x=df.index,
            y=avg_values,
            name=f'Avg Topic {topic}',
            visible=True,  # Set visibility to True for all average traces
            line=dict(color=topic_colors[topic], width=2)  # Set color based on topic
        )
        avg_traces.append(avg_trace)

    # Add average traces to the list of traces. This list is used to create our drop down
    all_traces = traces + avg_traces

    # Create dropdown options for conversations
    dropdown_options = []
    for convo in range(len(df.columns) // len(topics)):
        option = dict(
            label=f'Convo {convo}',
            method='update',
            args=[{'visible': [False] * len(all_traces)}]  
        )
        for i, trace in enumerate(all_traces):
            if trace.name.startswith(f'Convo {convo}'):
                option['args'][0]['visible'][i] = True  
        option['args'][0]['visible'][-len(topics):] = [True] * len(topics)  # Set visibility to True for average traces
        dropdown_options.append(option)

    # Create layout with dropdown menu
    layout = go.Layout(
        title='Topics Over Time',
        xaxis=dict(
            title='Chunk',
            tickmode='linear',  # Set tick mode to linear to ensure the x-axis i.e chunks contains only integers
            tick0=0,  # Start tick at 0
            dtick=1  # Set tick interval to 1
        ),
        yaxis=dict(title='Cosine Similarity (Topics Vs Chunk Embeddings)'),
        updatemenus=[
            dict(
                buttons=dropdown_options,
                direction='down',
                pad={'r': 10, 't': 10},
                showactive=True,
                x=0.05,
                xanchor='left',
                y=1.15,
                yanchor='top'
            )
        ]
    )

    # Create figure with traces and layout
    fig = go.Figure(data=all_traces, layout=layout)

    # Show the figure
    fig.show()

In [ ]:
# '''
# #THIS CODE WAS ORIGINALLY WRITTEN BY CHAT GPT, BECAUSE I WAS UNSURE ABOUT THE SYNTAX OF CREATING 
# INTERACTIVE VISUALS. I THEN MODIFIED AND SIMPLIFIED IT TO SUIT OUR NEEDS
# '''
# def plot_with_plotly(df):  

#     #Get the headings of the df eg. Convo 0 Topic 0, Convo 0 Topic 1. 
#     # I am not changing the original DF so that it is still readable to us when we write it to csv
#     convo_columns = df.columns 

#     # Extract conversation columns and topics. For e.g, if we have a heading as Convo 0 Topic 1
#     #col.split(' ')[-1] gives us the topic number as a string. This extracts the strings of unique topic numbers
#     topics = sorted(list(set([col.split(' ')[-1] for col in convo_columns])))

#     # Set colors for each topic dynamically - Not yet sure how it exactly works :(
#     num_topics = len(topics)
#     color_palette = plotly.colors.qualitative.D3
#     topic_colors = {topic: color_palette[i % len(color_palette)] for i, topic in enumerate(topics)}

#     #I HAVE VERFIED THIS LOGIC BY MANUALLY COMPUTING THE AVERAGES IN AN EXCEL SHEET
#     '''
#     ABOUT TRACES:

#     In Plotly, a "trace" refers to a data object that describes how to visualize a set of data points. 
#     It contains information about the type of plot (e.g., scatter, line, bar), the data to be plotted
#     (x, y, z coordinates or categorical values), and various styling options such as line color, marker
#     shape, and more.
    
#     Each trace represents a distinct set of data or a specific aspect of the visualization. For example, 
#     if you want to plot multiple lines on the same graph, you would create a separate trace for each line.

#     Traces are typically added to a data list and then passed to the `go.Figure` object in Plotly. 
#     The `go.Scatter` class is commonly used to create traces for line plots and scatter plots, but there
#     are other trace classes available for different types of plots, such as `go.Bar` for bar plots and
#     `go.Surface` for 3D surface plots.
#     '''

#     # Create traces for each conversation and topic
#     traces = []
#     for col in convo_columns:

#         # Splits the original column headsers. For e.g., if the heading is Convo 0 Topic 1,
#         convo = col.split(' ')[1] # This will be 0
#         topic = col.split(' ')[-1] # This will be 1

#         # Set visibility to True for convo 0 and False for others - This means the default value of the drop down is Convo 0
#         visible = True if convo == '0' else False  
        

#         #create a trace object
#         trace = go.Scatter(
#             x=df.index,
#             y=df[col],
#             name=col,
#             visible=visible,
#             line=dict(color=topic_colors[topic], width=1)  # Set color based on topic
#         )
#         traces.append(trace)

#     # Create trace for average of each topic
#     avg_traces = []
#     for topic in topics:
#         '''
#         get all the column headers which contain the current topic. For e.g. if the topic is '1', 
#         and there are 2 convos. This list contain the headers: Convo 0 Topic 1,Convo 1 Topic 1.
#         '''
#         topic_columns = [col for col in convo_columns if col.endswith(topic)]

#         '''
#         Average of Topics Across Convos for each chunk. Axis is 1 as averaging across columns.
#         Consider the following example df:

#         This df has 3 rows indicating there are 3 chunks of time.
#         data = {
#             'Convo 0 Topic 0': [0.476381, 0.795204, 0.559577],
#             'Convo 0 Topic 1': [0.466211, 0.498702, 0.768713],
#             'Convo 1 Topic 0': [0.699774, 0.734668, 0.767569],
#             'Convo 1 Topic 1': [0.449561, 0.484407, 0.523591]
#         }

#         If the current topic is topic 1, the code will average in the following way:

#         Average for Chunk 0: (0.466211 + 0.449561) / 2
#         Average for Chunk 1: (0.498702 + 0.484407) / 2
#         Average for Chunk 2: (0.768713+ 0.523591) / 2

#         Verfied by manual calculations in Excel
#         '''
#         avg_values = np.mean(df[topic_columns], axis=1)

#         #create the trace object
#         avg_trace = go.Scatter(
#             x=df.index,
#             y=avg_values,
#             name=f'Avg Topic {topic}',
#             visible=True,  # Set visibility to True for all average traces
#             line=dict(color=topic_colors[topic], width=2)  # Set color based on topic
#         )
#         avg_traces.append(avg_trace)

#     # Add average traces to the list of traces. This list is used to create our drop down
#     all_traces = traces + avg_traces

#     # Create dropdown options for conversations
#     dropdown_options = []
#     for convo in range(len(df.columns) // len(topics)):
#         option = dict(
#             label=f'Convo {convo}',
#             method='update',
#             args=[{'visible': [False] * len(all_traces)}]  
#         )
#         for i, trace in enumerate(all_traces):
#             if trace.name.startswith(f'Convo {convo}'):
#                 option['args'][0]['visible'][i] = True  
#         option['args'][0]['visible'][-len(topics):] = [True] * len(topics)  # Set visibility to True for average traces
#         dropdown_options.append(option)

#     # Create layout with dropdown menu
#     layout = go.Layout(
#         title='Topics Over Time',
#         xaxis=dict(
#             title='Chunk',
#             tickmode='linear',  # Set tick mode to linear to ensure the x-axis i.e chunks contains only integers
#             tick0=0,  # Start tick at 0
#             dtick=1  # Set tick interval to 1
#         ),
#         yaxis=dict(title='Cosine Similarity (Topics Vs Chunk Embeddings)'),
#         updatemenus=[
#             dict(
#                 buttons=dropdown_options,
#                 direction='down',
#                 pad={'r': 10, 't': 10},
#                 showactive=True,
#                 x=0.05,
#                 xanchor='left',
#                 y=1.15,
#                 yanchor='top'
#             )
#         ]
#     )

#     # Create figure with traces and layout
#     fig = go.Figure(data=all_traces, layout=layout)

#     # Show the figure
#     fig.show()

In [3037]:
def plot(df,num_chunks):

    # Set the x-axis tick locations to integers only
    plt.xticks(range(0, num_chunks + 1))

    # Extract the unique topics from the column names. This will extract the topic numbers from 0 to n as strings
    topics = set([col.split(' ')[-1] for col in df.columns])

    lines = []

    #get the list of topics
    for topic in topics:

        # Find columns related to the current topic - e.g. topic 0 in conv0,conv1 etc
        topic_columns = [col for col in df.columns if f'Topic {topic}' in col]

        # Get a color for the current topic
        color = plt.cm.get_cmap('tab10')(int(topic))

        # Plot lines for each column in the current topic
        for column in topic_columns:

            # Plot the line
            line, = plt.plot(df.index, df[column], label=column.split(' ')[0], color=color,alpha=0.5)

            # Get the last index of the DataFrame
            last_index = len(df.index) - 1

            # Get the x and y coordinates for the text label
            x = last_index
            y = df[column].iloc[last_index]

            # Add a text label to the line indicating the conversation number
            plt.text(x, y, f'Convo {column.split(" ")[1]}', color=color, fontsize=10, ha='left', va='bottom')
            lines.append(line)

    # Get the unique colors used in the line graph
    unique_colors = list(set([line.get_color() for line in lines]))

    # create manual tags - NOT THE BEST THING TO DO! NEED TO WORK ON IMPROVING THIS
    manual_tags = ['Topic 1', 'Topic 0']

    # Create custom legend elements
    legend_elements = [Line2D([0], [0], color=color, label=tag) for color, tag in zip(unique_colors, manual_tags)]
    
    # Add the custom legend
    plt.legend(handles=legend_elements)

    # Adding labels and title
    plt.xlabel('Chunk of Time')
    plt.ylabel('Topic Cosine Similarity w.r.t Conversation')
    plt.title('Topics Over Time')

    # Get the current figure
    fig = plt.gcf()

    # Set the new figure size (width, height) in inches
    fig.set_size_inches(8,8)
    
    # Display the chart
    plt.show()

In [3038]:
def plot_topics_over_time(df,num_chunks,num_topics):

    #get the total number of conversations. +1 as the range in python is 0 to n, exclusive of n
    total_convos = df['conversation_num'].max() + 1

    #fit the model
    model = fit_model(df['message'].tolist())

    #reduce the topics. We reduce to num_topics + 1 as there will always be a topic -1 with irrelevant topics
    model.reduce_topics(df['message'].tolist(),num_topics+1)

    #get the topics after reduction
    top_topics = get_top_topics(model,num_topics)

    #get the embeddings for the top topics
    embeddings_top_topics = get_embeddings_top_topics(df,num_topics)

    #remove the stopwords from the topics and recreate the topics without the stopwords
    topic_list = create_topics_without_stopwords(top_topics)

    #create an empty df. We will append columns for each Convo-Topic 
    df_for_plotting = None
    
    #iterate through all the conversations (e.g. 347 in Juries)
    for i in range (0,total_convos):

        print("Convo Number " +str(i))

        #filter the df for a specific conversation number
        df_convo = df[df['conversation_num'] == i]

        #create chunks by dividing the conversation into equal units of time
        create_chunks(df_convo,num_chunks)

        #get the embeddings for each chunk
        embeddings_per_chunk = get_embeddings_per_chunk(df_convo,num_chunks)

        #get the similarity between the topic embeddings and the chunk embeddings
        topics_per_chunk = get_similarity(embeddings_per_chunk,embeddings_top_topics)

        #convert the similarity matrix to a dataframe
        topics_df = pd.DataFrame(topics_per_chunk)

        new_column_headings = []

        #column heading for the current conversation. For e.g. 
        # if it is conversation 131 and we have 2 topics, the headings will be Convo 131 Topic 0, Convo 131 Topic 1
        for j in range(0,len(topic_list)):
            col_name = "Convo "+str(i)+" Topic "+str(j)
            new_column_headings.append(col_name)

        if i == 0:#if its the first conversation, we don't need to append the df to an existing df
            df_for_plotting = topics_df
            df_for_plotting.columns = new_column_headings
        else:
            topics_df.columns = new_column_headings
            df_for_plotting = df_for_plotting.join(topics_df)

    #print the top topics
    for i in range(0,len(topic_list)):
        print("Topic " +str(i)+": "+topic_list[i])

    #return the df: rows are the chunks, columns are Convo-Topics
    return df_for_plotting

In [3039]:
def bucket_df(df_for_plotting):
    return 0

In [3040]:
#testing on a small portion of the juries dataset
df = df_jury[df_jury['conversation_num'] < 5]
df_for_plotting = plot_topics_over_time(df,4,2)

Convo Number 0
Convo Number 1
Convo Number 2
Convo Number 3
Convo Number 4
Topic 0: think, asshole, husband, dont, help, shes, family, 
Topic 1: hello, hi, finally, whats, nan, impression, howdy, con, hey, conventionalmonkey, first, yeah, everyone, people, yes, mother, , , , , , , , , , , , , , 


In [3041]:
print(df_for_plotting)
df_for_plotting.to_csv('df_for_plotting.csv')

   Convo 0 Topic 0  Convo 0 Topic 1  Convo 1 Topic 0  Convo 1 Topic 1  \
0         0.376250         0.767710         0.626246         0.345042   
1         0.727422        -0.001185         0.686700         0.063915   
2         0.723617         0.065892         0.637921         0.054728   
3         0.540620         0.146340         0.626965         0.051445   

   Convo 2 Topic 0  Convo 2 Topic 1  Convo 3 Topic 0  Convo 3 Topic 1  \
0         0.621194         0.190039         0.697315         0.005550   
1         0.502369         0.114223         0.428036         0.208031   
2         0.541897         0.243802         0.586585         0.072432   
3         0.583172         0.156810         0.497650         0.163407   

   Convo 4 Topic 0  Convo 4 Topic 1  
0         0.381450         0.663487  
1         0.693473         0.020988  
2         0.610997         0.019834  
3         0.636432         0.014789  


In [3081]:
plot_with_plotly(df_for_plotting)

In [3043]:
# load the conversation level data for juries, and classify based on the best predictor score
jury_convo_df = pd.read_csv('../output/jury_output_conversation_level.csv')
jury_convo_df = jury_convo_df.head(5)

def get_buckets(df,performance_metric,threshold):
    df = df[[performance_metric]]

    #get the mean value of the performace metrix
    mean_value = df[performance_metric].mean()

    lower_threshold = mean_value - (mean_value * threshold)
    upper_threshold = mean_value + (mean_value * threshold)

    # Create a new column to store the classification
    df['label'] = ''

    # Classify the values
    for i, value in enumerate(df['majority_pct']):
        if value < lower_threshold:
            df.loc[i, 'label'] = 'Below Mean'
        elif value > upper_threshold:
            df.loc[i, 'label'] = 'Above Mean'
        else:
            df.loc[i, 'label'] = 'Around Mean'

    return df

In [3044]:
average_df = get_buckets(jury_convo_df,'majority_pct',0.1)
print(average_df)

   majority_pct        label
0      1.000000   Above Mean
1      0.600000   Below Mean
2      0.666667   Below Mean
3      0.750000  Around Mean
4      1.000000   Above Mean


In [3063]:
def bucket_based_average(df1,df2,num_topics):

    # Create a new DataFrame for aggregation
    df_agg = pd.DataFrame({'Chunk': range(len(df1))})

    # Iterate over the columns in df1 and assign labels based on matching convo numbers
    for col in df1.columns:
        convo = col.split(' ')[1]
        topic = col.split(' ')[-1]
        label = df2['label'][int(convo)]
        label_col = f'{label} Topic {topic}'
        if label_col not in df_agg.columns:
            df_agg[label_col] = 0.0
        df_agg[label_col] += df1[col]

    # Count the number of labels in each bucket
    label_counts = df2['label'].value_counts()
    
    #divide the aggregated values by the number of labels
    for topic in range(0,num_topics):
        topic_col = f'Topic {topic}'
        for label in label_counts.index:
            num_labels = label_counts[label]
            if num_labels > 0:
                df_agg[f'{label} {topic_col}'] /= num_labels 


    # Print the aggregated DataFrame
    df_agg.set_index('Chunk', inplace=True)

    return df_agg

In [3094]:
aggregated_df = bucket_based_average(df_for_plotting,average_df,2)
print(aggregated_df)
plot_with_plotly(aggregated_df)

       Above Mean Topic 0  Above Mean Topic 1  Below Mean Topic 0  \
Chunk                                                               
0                0.378850            0.715598            0.623720   
1                0.710448            0.009902            0.594535   
2                0.667307            0.042863            0.589909   
3                0.588526            0.080564            0.605068   

       Below Mean Topic 1  Around Mean Topic 0  Around Mean Topic 1  
Chunk                                                                
0                0.267540             0.697315             0.005550  
1                0.089069             0.428036             0.208031  
2                0.149265             0.586585             0.072432  
3                0.104127             0.497650             0.163407  
